In [ ]:
TERMS_ACCEPTED = False  # Set True only after you submit the authors' form and they accept it.
TERMS_FORM_URL = "https://docs.google.com/forms/d/e/1FAIpQLSdRRR3L5zAv6tQ_CKxmK4W96tAab_pfBu2EKAgQbeDVhmXagg/viewform"

print("Training is on Google Colab.")
print("This is Regime S: a documented subsample of the official training videos.")
print("It is not a reproduction of the paper's Table 1.")
print("Quality: H.264 QP 23 (c23) only. No raw, no c40, no YouTube scrape, no generator retraining.")
print("Subsample: 10 official training video ids, the 5 pairs frozen in configs/examples/regime-s-c23-pairs.json.")
print("8 of those videos fit the network. The other 2 stay out of the optimizer and are used only for the printed validation accuracy.")
print("Official validation ids and test ids are not downloaded.")
print("Face crop uses OpenCV YuNet 2023mar, not the paper Face2Face tracker. The enlargement factor is still 1.3.")
print("Before Run all: Runtime -> Change runtime type -> T4 GPU.")
print("Open the FaceForensics++ access form. The authors email download-FaceForensics.py after they accept it:")
print(TERMS_FORM_URL)
if not TERMS_ACCEPTED:
    raise SystemExit("Stopped until TERMS_ACCEPTED is True.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jassimarjeet/Verishield.git"
REPO = Path("/content/Verishield")
if REPO.exists():
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"])
else:
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO)])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
print("repo", REPO)


In [ ]:
from pathlib import Path

from src.data.ffpp_download import author_download_args, load_author_script

DOWNLOAD_SCRIPT = "/content/download-FaceForensics.py"
TOS_CONFIRMED = False  # Set True only after you read the terms URL printed below.
SERVER = "EU"  # If this server fails, set SERVER = "EU2" and run this cell again.

script = Path(DOWNLOAD_SCRIPT)
if not script.is_file():
    raise SystemExit(
        "Upload the script the authors emailed to /content/download-FaceForensics.py "
        "using the Files panel. Do not replace it with a YouTube downloader."
    )
module = load_author_script(script)
args = author_download_args(module, script, Path("/content/ffpp"), SERVER)
print("Authors' terms URL from their script:")
print(args.tos_url)
print("This run uses -c c23 -t videos for original, Deepfakes, Face2Face, FaceSwap, and NeuralTextures.")
print("It does not use -d all, original_youtube_videos, raw, or c40.")
if not TOS_CONFIRMED:
    raise SystemExit("Stopped until TOS_CONFIRMED is True.")


In [ ]:
from pathlib import Path

from src.data.ffpp_download import download_c23_subset

download_c23_subset(
    script_path=Path(DOWNLOAD_SCRIPT),
    output_dir=Path("/content/ffpp"),
    server=SERVER,
    terms_accepted=TERMS_ACCEPTED,
    tos_confirmed=TOS_CONFIRMED,
)


In [ ]:
from pathlib import Path

from src.train.regime_s import train_regime_s

checkpoint = Path("/content/verishield-checkpoints/regime-s-c23/best.pt")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    checkpoint = Path("/content/drive/MyDrive/verishield/checkpoints/regime-s-c23/best.pt")
    print("Checkpoint path is on My Drive. A Colab ephemeral disk is not the backup.")
except Exception as exc:
    print("Drive was not mounted:", exc)
    print(checkpoint)
    print("That path is on the Colab ephemeral disk. It is not the backup.")

history = train_regime_s(
    data_root=Path("/content/ffpp"),
    checkpoint_path=checkpoint,
    pretrained=True,
    seed=0,
)
print("Computed history from this session. These numbers are not filled in ahead of the run:")
for row in history:
    print(row)
